In [ ]:
# Configure every external directory here before executing the notebook.
from pathlib import Path
import os
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DIR = Path(os.environ.get('RIDESHARE_RAW_DIR', str(PROJECT_DIR / 'data/raw/fhvhv')))
PROCESSED_DIR = Path(os.environ.get('RIDESHARE_PROCESSED_DIR', str(PROJECT_DIR / 'data/processed')))
ZONE_DIR = Path(os.environ.get('RIDESHARE_ZONE_DIR', str(PROJECT_DIR / 'data/reference/taxi_zones')))
WEATHER_DIR = Path(os.environ.get('RIDESHARE_WEATHER_DIR', str(PROJECT_DIR / 'data/reference/weather_hourly')))
ANALYSIS_DIR = PROJECT_DIR / 'outputs/demand_analysis'
# Spark users: configure JAVA_HOME and, on Windows when required, HADOOP_HOME externally.

# All directories are configured here. Hide this cell to hide local paths.
from pathlib import Path
DATA_DIR = PROCESSED_DIR
REPAIR_DIR = DATA_DIR / 'repair_2024_2025'
ZONE_DIR = ZONE_DIR
RESULTS_DIR = ANALYSIS_DIR
YEARS = [2021, 2022, 2023, 2024, 2025]
# Read validated repair candidates in place; never promote or overwrite originals.
USE_VALIDATED_REPAIRS = True
SAVE_FIGURES = True


# Baseline pickup demand

## Scope and interpretation

This analysis uses cleaned **pickup totals from January 2021 through December 2025**, across all recorded HVFHV platforms. It describes observed trip activity, not requests, vehicle supply, duration or a causal policy effect. All 60 monthly inputs must pass coverage checks before cross-year comparisons. Calendar-day exposure is used for normalized volume; missing inputs are never interpreted as zero demand. Local hour labels combine repeated daylight-saving hours.

All directories are configured in the first code cell. The map joins on LocationID and dissolves multipart zone records. Unmapped IDs remain in citywide totals and share denominators.


In [ ]:
import calendar
import json
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
db = duckdb.connect()
db.execute("SET threads=1")
db.execute("SET memory_limit='256MB'")
db.execute("SET preserve_insertion_order=false")
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 110})
def finish(fig, name):
    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(RESULTS_DIR / (name + '.png'), dpi=180, bbox_inches='tight')
    plt.show()


In [ ]:
NOTEBOOK_ID = '03_baseline'
DIRECTIONS = ['pickup']


In [ ]:
# Check each monthly pickup table before including it in the analysis.
manifest, daily_frames, zone_frames, profile_frames = [], [], [], []
for year in YEARS:
    for month in range(1, 13):
        folder = DATA_DIR / f'fhvhv_by_zone_hour_{year}_{month:02d}.parquet'
        parts = sorted(folder.glob('*.parquet'))
        assert parts and (folder / '_SUCCESS').exists(), f'Incomplete input: {folder.name}'
        db.read_parquet([str(p) for p in parts]).create_view('source', replace=True)
        bad = db.execute(f"""SELECT count(*) FROM source WHERE
            PULocationID IS NULL OR date IS NULL OR hour IS NULL
            OR hour NOT BETWEEN 0 AND 23 OR trip_count IS NULL OR trip_count <= 0
            OR year(date) != {year} OR month(date) != {month}""").fetchone()[0]
        duplicates = db.execute("""SELECT count(*) FROM
            (SELECT 1 FROM source GROUP BY PULocationID, date, hour HAVING count(*) > 1)""").fetchone()[0]
        dates = db.execute('SELECT count(DISTINCT date) FROM source').fetchone()[0]
        assert bad == duplicates == 0, f'Invalid rows/keys: {folder.name}'
        assert dates == calendar.monthrange(year, month)[1], f'Incomplete dates: {folder.name}'
        manifest.append(dict(direction='pickup', year=year, month=month,
                             status='included', path=str(folder)))
        queries = [
            (daily_frames, 'SELECT date, sum(trip_count) trips FROM source GROUP BY date'),
            (zone_frames, 'SELECT PULocationID zone_id, sum(trip_count) trips FROM source GROUP BY PULocationID'),
            (profile_frames, 'SELECT dayofweek(date) dow, hour AS "hour", sum(trip_count) trips FROM source GROUP BY dayofweek(date), hour'),
        ]
        for frames, sql in queries:
            frame = db.execute(sql).df()
            frame = frame.assign(direction='pickup', year=year, month=month)
            frames.append(frame)
manifest = pd.DataFrame(manifest)
assert len(manifest) == 60
daily = pd.concat(daily_frames, ignore_index=True)
daily['date'] = pd.to_datetime(daily.date)
zone_month = pd.concat(zone_frames, ignore_index=True)
profiles = pd.concat(profile_frames, ignore_index=True)
assert daily.date.between('2021-01-01', '2025-12-31').all()
manifest.to_csv(RESULTS_DIR / f'{NOTEBOOK_ID}_input_manifest.csv', index=False)
daily.to_csv(RESULTS_DIR / f'{NOTEBOOK_ID}_daily_totals.csv', index=False)
display(manifest.groupby('year').size().rename('complete_months'))


## Daily pickup trend
Daily totals and seven-day rolling means summarize the 2021–2025 series. The rolling mean requires seven observed days.


In [ ]:
# Uses DATA_DIR, REPAIR_DIR, and manifest from the notebook.
# Run after the data-loading cell.

hourly_frames = []

for item in manifest.loc[manifest.status.eq('included')].itertuples():
    folder = Path(item.path)
    hour_col = 'hour' if item.direction == 'pickup' else 'do_hour'

    db.read_parquet(
        [str(p) for p in folder.glob('*.parquet')]
    ).create_view('hourly_source', replace=True)

    frame = db.execute(f'''
        SELECT date, "{hour_col}" AS "hour",
               SUM(trip_count) AS trips
        FROM hourly_source
        GROUP BY date, "{hour_col}"
    ''').df()

    frame['direction'] = item.direction
    hourly_frames.append(frame)

hourly = pd.concat(hourly_frames, ignore_index=True)
hourly['timestamp'] = (
    pd.to_datetime(hourly['date'])
    + pd.to_timedelta(hourly['hour'], unit='h')
)

fig, axes = plt.subplots(1, 1, figsize=(16, 5), sharex=True)
axes = np.atleast_1d(axes)

for ax, direction in zip(axes, DIRECTIONS):
    s = (
        hourly.loc[hourly.direction.eq(direction)]
        .set_index('timestamp')['trips']
        .sort_index()
        .asfreq('h')
    )

    ax.plot(
        s.index, s / 1e3,
        color='#444444', lw=0.35, alpha=0.65,
        label='Hourly trips'
    )
    ax.plot(
        s.index, s.rolling(168, min_periods=168).mean() / 1e3,
        color='#D55E00', lw=1.5,
        label='7-day rolling mean (168 hours)'
    )

    ax.set(
        title=direction.title(),
        ylabel='Thousand trips/hour'
    )
    ax.legend(loc='upper left')

axes[-1].set_xlabel('Date')
fig.suptitle('Hourly Pickup Demand, 2021–2025', fontsize=18)
# Limit the displayed year ticks to the descriptive study period.
for ax in axes:
    ax.set_xlim(pd.Timestamp('2021-01-01'), pd.Timestamp('2025-12-31 23:00'))
    ax.set_xticks(pd.date_range('2021-01-01', '2025-01-01', freq='YS'))
    ax.set_xticklabels([str(y) for y in YEARS])
finish(fig, '03_hourly_demand_entire_period')

## Daily pickup trend
Daily totals and seven-day rolling means summarize the 2021–2025 series. The rolling mean requires seven observed days.


In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(16, 5), sharex=True)
axes = np.atleast_1d(axes)

for ax, direction in zip(axes, DIRECTIONS):
    s = (
        daily[daily.direction == direction]
        .set_index("date").trips.sort_index().asfreq("D")
    )

    ax.plot(
        s.index, s / 1e3,
        color="black", alpha=0.7, linewidth=1,
        label="Daily", zorder=1
    )
    ax.plot(
        s.index, s.rolling(7, min_periods=7).mean() / 1e3,
        color="#D55E00", alpha=1, linewidth=2,
        label="7-day mean", zorder=2
    )

    ax.set(title=direction.title(), ylabel="Thousand trips/day")
    ax.legend()

fig.suptitle("Daily Pickup Demand, 2021–2025", fontsize=16)
# Limit the displayed year ticks to the descriptive study period.
for ax in axes:
    ax.set_xlim(pd.Timestamp('2021-01-01'), pd.Timestamp('2025-12-31 23:00'))
    ax.set_xticks(pd.date_range('2021-01-01', '2025-01-01', freq='YS'))
    ax.set_xticklabels([str(y) for y in YEARS])
finish(fig, "03_daily_demand")

In [ ]:
zero_pickups = daily.loc[(daily["direction"] == "pickup") & (daily["trips"] <200000)].sort_values("date")

display(zero_pickups)

## Monthly demand and calendar exposure
Show both totals and trips per observed calendar day. Gaps remain gaps.

In [ ]:
monthly = daily.groupby(['direction','year','month'],as_index=False).agg(trips=('trips','sum'), days=('date','nunique'))
monthly['trips_per_day'] = monthly.trips / monthly.days
fig, axes = plt.subplots(1,2,figsize=(15,5))
for direction in DIRECTIONS:
    m = monthly[monthly.direction==direction].copy()
    m['date'] = pd.to_datetime(dict(year=m.year,month=m.month,day=1))
    start, end = m.date.min(), m.date.max()
    m = m.set_index('date').reindex(pd.date_range(start,end,freq='MS'))
    axes[0].plot(m.index,m.trips/1e6,label=direction)
    axes[1].plot(m.index,m.trips_per_day/1e3,label=direction)
axes[0].set(title='Monthly totals',ylabel='Million trips')
axes[1].set(title='Calendar-day normalized demand',ylabel='Thousand trips/day')
for ax in axes: ax.legend()
finish(fig,'03_monthly_demand')
display(monthly)
monthly.to_csv(RESULTS_DIR/'03_monthly_totals.csv',index=False)


## Comparable-month year summaries
Use months present in every selected year and both directions. This avoids comparing a partial year with a full year.

In [ ]:
coverage = monthly.groupby(['direction','year']).month.apply(set)
common_months = sorted(set.intersection(*coverage.tolist()))
comparison = monthly[monthly.month.isin(common_months)].groupby(['direction','year']).agg(trips=('trips','sum'),days=('days','sum'))
comparison['trips_per_day'] = comparison.trips/comparison.days
print('Common months:',common_months)
display(comparison)
fig,ax=plt.subplots()
comparison.trips_per_day.unstack('direction').div(1e3).plot.bar(ax=ax)
ax.set(ylabel='Thousand trips/day',title='Demand in common observed months')
finish(fig,'03_common_month_year_comparison')


## Data issues and findings

This baseline uses complete pickup months in 2021–2025. Daily demand is descriptive; forecasting targets hourly zonal pickups. Missing source months must not be filled with zeros. Review low-demand hours against calendar and input coverage before any imputation. See the temporal/spatial findings in notebooks 04 and 05 for quantified results.
